# Image Colorizer — Self-Attention + COCO (Demo)

End-to-end demo of the trainable colorizer: set up data, (optionally) train a few epochs, load a checkpoint, colorize sample images, and report PSNR/SSIM.

Run this on a GPU runtime (Colab: *Runtime → Change runtime type → GPU*).

## 1. Setup

In [ ]:
!pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available())

## 2. Download a COCO subset

The `val2017` split (~5k images) is small enough for a quick demo. Use `--max-samples` to grab fewer.

In [ ]:
!python download_data.py --data-root data/coco --max-samples 2000

## 3. Train a few epochs

For a real model, raise `--epochs`. Re-running this cell resumes from the latest checkpoint.

In [ ]:
!python train.py --data-root data/coco --epochs 5 --batch-size 16

## 4. Load the checkpoint and colorize samples

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from skimage import color

from config import get_config
from dataset import ColorizationDataset, denormalize_to_rgb
from model import build_model
from metrics import image_psnr_ssim

cfg = get_config()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model(cfg).to(device)
ck = torch.load('checkpoints/latest.pt', map_location=device)
model.load_state_dict(ck['model'])
model.eval()
print('Loaded checkpoint at step', ck.get('global_step'))

val_ds = ColorizationDataset(cfg.data_root, 'val', cfg.image_size,
                             cfg.val_fraction, cfg.split_seed)

n = 4
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
for i in range(n):
    L, ab_gt = val_ds[i]
    with torch.no_grad():
        ab_pred = model(L[None].to(device))[0].cpu().numpy()
    L_np = L.numpy()
    gray = np.clip((L_np[0] + 1) * 50, 0, 100)
    gray_rgb = color.lab2rgb(np.stack([gray, np.zeros_like(gray), np.zeros_like(gray)], -1))
    gt = denormalize_to_rgb(L_np, ab_gt.numpy())
    pred = denormalize_to_rgb(L_np, ab_pred)
    psnr_v, ssim_v = image_psnr_ssim(gt, pred)
    axes[i, 0].imshow(gray_rgb); axes[i, 0].set_title('input (L)')
    axes[i, 1].imshow(pred); axes[i, 1].set_title(f'predicted\nPSNR {psnr_v:.1f}  SSIM {ssim_v:.2f}')
    axes[i, 2].imshow(gt); axes[i, 2].set_title('ground truth')
    for j in range(3):
        axes[i, j].axis('off')
plt.tight_layout(); plt.show()

## 5. Aggregate PSNR / SSIM on the val split

In [ ]:
from torch.utils.data import DataLoader
from metrics import evaluate

val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, num_workers=cfg.num_workers)
res = evaluate(model, val_loader, device, max_batches=10)
print(f"Mean PSNR: {res['psnr']:.4f} dB   Mean SSIM: {res['ssim']:.4f}   (n={res['n']})")